# 05 – Modeling
Trains and compares Logistic Regression, Random Forest, XGBoost, and LightGBM on the processed dataset.

In [ ]:
import sys; sys.path.insert(0, '..')
import json
import time
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, classification_report, roc_curve
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
df = pd.read_parquet('../data/processed/accidents_features.parquet')
with open('../models/feature_metadata.json') as f:
    meta = json.load(f)
features = [f for f in meta['features'] if f in df.columns]
X = df[features].fillna(0)
y = df['target']
print(f'Dataset: {X.shape} | Class balance: {y.value_counts().to_dict()}')

In [ ]:
# Use a sample for notebook speed (full training done via train_model.py)
SAMPLE = 200_000
idx = np.random.RandomState(42).choice(len(X), min(SAMPLE, len(X)), replace=False)
X_s, y_s = X.iloc[idx], y.iloc[idx]
X_train, X_test, y_train, y_test = train_test_split(X_s, y_s, test_size=0.2, random_state=42, stratify=y_s)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

## Logistic Regression (Baseline)

In [ ]:
t0 = time.time()
lr = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(class_weight='balanced', max_iter=500, random_state=42))])
lr.fit(X_train, y_train)
lr_prob = lr.predict_proba(X_test)[:, 1]
lr_auc = roc_auc_score(y_test, lr_prob)
print(f'Logistic Regression | AUC={lr_auc:.4f} | Time={time.time()-t0:.1f}s')
print(classification_report(y_test, lr.predict(X_test), target_names=['Low', 'High']))

## Random Forest

In [ ]:
t0 = time.time()
rf = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced', n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
rf_prob = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_prob)
print(f'Random Forest | AUC={rf_auc:.4f} | Time={time.time()-t0:.1f}s')
print(classification_report(y_test, rf.predict(X_test), target_names=['Low', 'High']))

## XGBoost

In [ ]:
t0 = time.time()
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, scale_pos_weight=4,
                    tree_method='hist', device='cpu', random_state=42, verbosity=0)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xgb_prob = xgb.predict_proba(X_test)[:, 1]
xgb_auc = roc_auc_score(y_test, xgb_prob)
print(f'XGBoost | AUC={xgb_auc:.4f} | Time={time.time()-t0:.1f}s')
print(classification_report(y_test, xgb.predict(X_test), target_names=['Low', 'High']))

## LightGBM

In [ ]:
t0 = time.time()
lgbm = LGBMClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, class_weight='balanced',
                      n_jobs=-1, random_state=42, verbose=-1)
lgbm.fit(X_train, y_train)
lgbm_prob = lgbm.predict_proba(X_test)[:, 1]
lgbm_auc = roc_auc_score(y_test, lgbm_prob)
print(f'LightGBM | AUC={lgbm_auc:.4f} | Time={time.time()-t0:.1f}s')
print(classification_report(y_test, lgbm.predict(X_test), target_names=['Low', 'High']))

## ROC Curve Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, prob, auc in [('Logistic Regression', lr_prob, lr_auc), ('Random Forest', rf_prob, rf_auc),
                         ('XGBoost', xgb_prob, xgb_auc), ('LightGBM', lgbm_prob, lgbm_auc)]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--', lw=1)
ax.set(xlabel='False Positive Rate', ylabel='True Positive Rate', title='ROC Curves Comparison')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Feature Importance (XGBoost)

In [ ]:
imp = pd.Series(xgb.feature_importances_, index=features).sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(10, 6))
imp.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('XGBoost Feature Importance (Top 20)')
plt.tight_layout()
plt.show()

## Learning Curves (Random Forest)

In [ ]:
train_sizes, train_scores, val_scores = learning_curve(
    RandomForestClassifier(n_estimators=50, max_depth=8, class_weight='balanced', n_jobs=-1, random_state=42),
    X_train, y_train, cv=3, scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 6), n_jobs=-1
)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Train AUC', color='steelblue')
ax.fill_between(train_sizes, train_scores.mean(1)-train_scores.std(1), train_scores.mean(1)+train_scores.std(1), alpha=0.2, color='steelblue')
ax.plot(train_sizes, val_scores.mean(axis=1), 'o-', label='Val AUC', color='coral')
ax.fill_between(train_sizes, val_scores.mean(1)-val_scores.std(1), val_scores.mean(1)+val_scores.std(1), alpha=0.2, color='coral')
ax.set(title='Learning Curves – Random Forest', xlabel='Training Size', ylabel='AUC')
ax.legend()
plt.tight_layout()
plt.show()